# Control run: V1's data (burak29) on V2's model and recipe

Same base model (Qwen3.5-0.8B), same LoRA, same masking and trainer as `v2_verified-dataset_qwen3.5-0.8b_lora.ipynb`.
**Only the training data changes**: the public burak29 dataset (1,030 natural-language → ffmpeg pairs) converted to the
`run`-tool chat format, instead of the verified Clipwright data. Evaluating both adapters with the same harness shows how
much of V2's result comes from the data rather than the model.

Differences from the V2 notebook, all forced by the data:
- **Data cell**: burak29 rows converted to `system → user (+ Files list) → run tool call → tool result → "Done."`.
  Paths are mapped to the sandbox layout (`-i video.mp4` → `in/video.mp4`, output → `out/`). Rows that can't run in the
  sandbox (streaming URLs, devices, `-f null -` analysis) are skipped: 990 of 1,030 kept.
  The tool result is a plain `{"exit_code": 0}`: burak29 commands were never executed, so there is no real output to show.
- **Epochs: 3** (V1's original setting). 1 epoch of 990 rows would be only ~60 optimizer steps.
- **Eval/save every 20 steps** instead of 100, since the whole run is ~180 steps.
- No weight-0 rows exist in this data, so that mask-demo cell is gone.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!pip install -q -U transformers peft trl datasets accelerate wandb

In [ ]:
!pip install -q flash-linear-attention

In [ ]:
!pip uninstall -y torchao

In [ ]:
!pip install causal_conv1d

In [ ]:
!pip install -q liger-kernel

In [ ]:
MODEL      = "Qwen/Qwen3.5-0.8B"
DTYPE      = "bf16"          # L4
ATTN       = "sdpa"
MAX_LEN = 8192            # confirm with the memory test below; fall back to 6144 / 4096
N_ROWS     = None            # ALL rows
NUM_EPOCHS = 3             # V1 used 3 epochs; 1 epoch of 990 rows is only ~60 steps
OUT_DIR    = "/content/drive/MyDrive/data-sft/runs/qwen08b-burak29"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
from datasets import Dataset, load_dataset

# Same system prompt and tool schema as the verified data (build.SYSTEM, sandbox.TOOL_SCHEMA)
SYSTEM = "You are a media-processing assistant working in a sandbox with ffmpeg, ffprobe, ImageMagick (magick), SoX and ExifTool. The user's files are in in/ (read-only) and are listed with their properties under 'Files:'. Write every result to out/; that folder is returned to the user. Use the run tool to execute commands. Before each command, briefly explain your reasoning, based on the actual file properties (e.g. copy streams instead of re-encoding when the codec already fits). Quote paths that contain spaces. If a command fails, read the error and fix it. When done, reply with a short summary of the output files. If a request is ambiguous or impossible with these tools, ask or explain instead of guessing."
TOOL = {"type": "function", "function": {"name": "run", "description": "Run one media CLI command (ffmpeg, ffprobe, magick, sox, soxi, exiftool) in the job directory. Inputs are in in/ (read-only), write results to out/. Chain steps with &&. Set foreach to a glob to run the command once per matching file, with {in} = the file path and {stem} = its name without extension.", "parameters": {"type": "object", "properties": {"cmd": {"type": "string", "description": "The command line."}, "foreach": {"type": "string", "description": "Optional glob, e.g. in/*.mov"}}, "required": ["cmd"]}}}

import json, re, shlex

MEDIA = re.compile(r"[\w.%*/-]+\.[A-Za-z0-9]{2,5}$")  # looks like a file path

def convert(nl, cmd):
    """burak29 pair -> run-tool chat in the same layout as the verified data. None if it can't be mapped."""
    try:
        t = shlex.split(cmd)
    except ValueError:
        return None
    if not t or t[0] != "ffmpeg":
        return None
    ins = []
    for i, tok in enumerate(t[:-1]):
        if tok == "-i" and t[i + 1] != "-" and ":" not in t[i + 1] and not t[i + 1].startswith("/"):  # skip devices/URLs/stdin
            ins.append(t[i + 1])
            t[i + 1] = "in/" + t[i + 1]
    out = t[-1]
    if not MEDIA.match(out) or out in ins:
        return None
    t[-1] = "out/" + out
    new = shlex.join(t)
    for f in ins:  # filter args that name an input, e.g. subtitles=subtitles.srt, movie=logo.png
        new = re.sub(rf"(?<=[=']){re.escape(f)}\b", "in/" + f, new)
    files = "\n".join(f"- in/{f}" for f in dict.fromkeys(ins)) or "(none)"
    return [
        {"role": "user", "content": f"{nl}\n\nFiles:\n{files}"},
        {"role": "assistant", "content": "", "tool_calls": [
            {"type": "function", "function": {"name": "run", "arguments": {"cmd": new}}}]},
        {"role": "tool", "name": "run", "content": json.dumps({"exit_code": 0})},
        {"role": "assistant", "content": f"Done.\n- out/{out}"},
    ]

raw = load_dataset("burak29/Natural_Language_to_Ffmpeg_Commands", split="train")
rows = []
for r in raw:
    msgs = convert(r["natural_language"], r["ffmpeg_command"])
    if msgs is None:
        continue
    msgs = [{"role": "system", "content": SYSTEM}] + msgs
    for m in msgs:                      # tool-call arguments as JSON strings, same as the V2 notebook (Arrow note)
        if m.get("tool_calls"):
            m["tool_calls"] = [{**tc, "function": {**tc["function"],
                                "arguments": json.dumps(tc["function"]["arguments"], ensure_ascii=False)}}
                               for tc in m["tool_calls"]]
    rows.append({"messages": msgs, "tools": [TOOL]})
print(f"converted {len(rows)} of {len(raw)} rows")
if N_ROWS:
    rows = rows[:N_ROWS]

ds = Dataset.from_list(rows)
split = ds.train_test_split(test_size=0.02, seed=42)   # ~2% for validation loss, as in V2
train_ds, val_ds = split["train"], split["test"]

print(train_ds, val_ds)
print(json.dumps(train_ds[0]["messages"], indent=1, ensure_ascii=False)[:2000])


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
print(type(tokenizer).__name__, "| vocab:", len(tokenizer))

In [ ]:
def for_template(messages):
    """Undo storage tricks before rendering: arguments back to dicts,
    and drop the None fields Arrow added (tool_calls=None, weight=None, ...)."""
    out = []
    for m in messages:
        m = {k: v for k, v in m.items() if v is not None}        # remove Arrow's None padding
        if m.get("tool_calls"):
            m["tool_calls"] = [
                {**tc, "function": {**tc["function"],
                                    "arguments": json.loads(tc["function"]["arguments"])}}
                for tc in m["tool_calls"]
            ]
        out.append(m)
    return out

row = train_ds[0]
text = tokenizer.apply_chat_template(
    for_template(row["messages"]),
    tools=row["tools"],
    tokenize=False,
    enable_thinking=False,
)
print(text)

In [ ]:
import re

# One assistant turn: the header, the empty think block, then the part we train on (up to and incl. <|im_end|>)
ASSISTANT_TURN = re.compile(
    r"<\|im_start\|>assistant\n(?:<think>\n\n</think>\n\n)?(.*?<\|im_end\|>)", re.DOTALL)

def tokenize_row(row):
    msgs = for_template(row["messages"])
    text = tokenizer.apply_chat_template(msgs, tools=row["tools"], tokenize=False, enable_thinking=False)

    # character spans of each assistant turn's trainable part, in order
    spans = [m.span(1) for m in ASSISTANT_TURN.finditer(text)]
    weights = [m.get("weight", 1) for m in msgs if m["role"] == "assistant"]
    assert len(spans) == len(weights), f"found {len(spans)} assistant turns, expected {len(weights)}"
    spans = [s for s, w in zip(spans, weights) if w != 0]          # drop weight-0 turns

    enc = tokenizer(text, return_offsets_mapping=True, add_special_tokens=False)
    labels = []
    for tok_id, (a, b) in zip(enc["input_ids"], enc["offset_mapping"]):
        inside = any(start <= a and b <= end for start, end in spans)
        labels.append(tok_id if inside else -100)

    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": labels}


def show_mask(row):
    out = tokenize_row(row)
    pieces, inside = [], False
    for tok_id, lab in zip(out["input_ids"], out["labels"]):
        t = tokenizer.decode([tok_id])
        if lab != -100 and not inside: pieces.append("[["); inside = True
        if lab == -100 and inside:     pieces.append("]]"); inside = False
        pieces.append(t)
    if inside: pieces.append("]]")
    print("".join(pieces))
    n = sum(l != -100 for l in out["labels"])
    print(f"\n--- {len(out['input_ids'])} tokens, {n} trained ({n/len(out['input_ids']):.0%})")

show_mask(train_ds[0])

In [ ]:
cols = train_ds.column_names                      # ["messages", "tools"]: replaced by token columns
train_tok = train_ds.map(tokenize_row, remove_columns=cols, num_proc=2)
val_tok   = val_ds.map(tokenize_row,   remove_columns=cols, num_proc=2)

before = len(train_tok)
train_tok = train_tok.filter(lambda r: len(r["input_ids"]) <= MAX_LEN)
val_tok   = val_tok.filter(lambda r: len(r["input_ids"]) <= MAX_LEN)
print(f"train {len(train_tok)} (dropped {before - len(train_tok)} over {MAX_LEN} tokens), val {len(val_tok)}")

lens = sorted(len(x) for x in train_tok["input_ids"])
print("tokens: median", lens[len(lens)//2], "| p99", lens[int(len(lens)*.99)], "| max", lens[-1])

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

from liger_kernel.transformers.monkey_patch import MODEL_TYPE_TO_APPLY_LIGER_FN
MODEL_TYPE_TO_APPLY_LIGER_FN["qwen3_5_text"]()

dtype = torch.bfloat16 if DTYPE == "bf16" else torch.float16
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=dtype, attn_implementation=ATTN, device_map="cuda")

model.gradient_checkpointing_enable()   # trade compute for memory (see below)
model.enable_input_require_grads()      # needed when gradient checkpointing + LoRA are combined
model.config.use_cache = False          # the generation cache is useless during training

lora = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05,
    target_modules="all-linear",        # LoRA on every linear layer (attention + MLP)
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()      # expect roughly 1-2% trainable

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

steps_total = len(train_tok) // 16 * NUM_EPOCHS
print("total optimizer steps:", steps_total)

args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=int(0.03 * steps_total),
    num_train_epochs=NUM_EPOCHS,
    logging_steps=10,
    eval_strategy="steps", eval_steps=20,     # whole run is ~180 steps
    save_strategy="steps", save_steps=20,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    bf16=(DTYPE == "bf16"), fp16=(DTYPE == "fp16"),
    train_sampling_strategy="group_by_length",
    report_to="none",
    per_device_eval_batch_size=1,
    prediction_loss_only=True,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_tok, eval_dataset=val_tok,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),
)
trainer.train()

In [ ]:
!ls {OUT_DIR}

In [ ]:
trainer.save_model(f"{OUT_DIR}/final")
tokenizer.save_pretrained(f"{OUT_DIR}/final")

In [ ]:
model.eval()
test = for_template(val_ds[0]["messages"])[:2]          # system + user only
prompt = tokenizer.apply_chat_template(test, tools=val_ds[0]["tools"], tokenize=False,
                                        add_generation_prompt=True, enable_thinking=False)

ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")   # text → token ids on GPU
out = model.generate(**ids, max_new_tokens=300, do_sample=False)                     # model writes up to 300 tokens

print(tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=False))  # only the new part

Download `{OUT_DIR}/final` (the adapter, ~100 MB). The GGUF conversion runs on the Mac, same as for V2.